# Avaliação de previsões (janela deslizante)

Notebook para calcular MAE/MSE, box-plots e gráficos das séries (observado vs predito) com previsões em CSV na pasta `previsoes/`.

In [ ]:
from pathlib import Path
from typing import Dict, List
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)

In [ ]:
PREVISOES_DIR = Path('../previsoes')
CSV_PATTERN = '*.csv'
REAL_CANDIDATES = ['y_true','real','observado','actual','target']
PRED_CANDIDATES = ['y_pred','pred','previsao','forecast']
SERIE_CANDIDATES = ['serie','series','canal','channel','id_serie']
TEMPO_CANDIDATES = ['timestamp','data','date','ds','tempo']
LOTE_CANDIDATES = ['lote','batch','janela','window','forecast_window','origem_previsao']

In [ ]:
def resolve_column(df: pd.DataFrame, candidates: List[str], required: bool = True):
    lower_map = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    if required:
        raise ValueError(f'Nenhuma coluna encontrada para {candidates}. Colunas: {list(df.columns)}')
    return None

def load_forecasts_csvs(previsoes_dir: Path, pattern: str = '*.csv') -> pd.DataFrame:
    files = sorted(previsoes_dir.glob(pattern))
    if not files:
        raise FileNotFoundError(f'Nenhum CSV encontrado em: {previsoes_dir.resolve()}')
    frames = []
    for fp in files:
        dfi = pd.read_csv(fp)
        dfi['arquivo_origem'] = fp.name
        frames.append(dfi)
    df = pd.concat(frames, ignore_index=True)

    col_real = resolve_column(df, REAL_CANDIDATES)
    col_pred = resolve_column(df, PRED_CANDIDATES)
    col_serie = resolve_column(df, SERIE_CANDIDATES)
    col_tempo = resolve_column(df, TEMPO_CANDIDATES)
    col_lote = resolve_column(df, LOTE_CANDIDATES)

    out = df[[col_serie, col_tempo, col_lote, col_real, col_pred, 'arquivo_origem']].copy()
    out.columns = ['serie', 'timestamp', 'lote', 'y_true', 'y_pred', 'arquivo_origem']
    out['timestamp'] = pd.to_datetime(out['timestamp'])
    return out.sort_values(['serie', 'timestamp', 'lote']).reset_index(drop=True)

def add_prediction_views(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby(['serie', 'timestamp'], sort=False)
    first = g.first(numeric_only=False)['y_pred'].rename('y_pred_first')
    median = g['y_pred'].median().rename('y_pred_median')
    last = g.last(numeric_only=False)['y_pred'].rename('y_pred_last')
    y_true = g.first(numeric_only=False)['y_true'].rename('y_true')
    views = pd.concat([y_true, first, median, last], axis=1).reset_index()
    for p in ['first', 'median', 'last']:
        views[f'resid_{p}'] = views['y_true'] - views[f'y_pred_{p}']
        views[f'ae_{p}'] = views[f'resid_{p}'].abs()
        views[f'se_{p}'] = views[f'resid_{p}'] ** 2
    return views

def rolling_metrics_by_batch(df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    out = {}
    for p in ['first', 'median', 'last']:
        tmp = df.copy()
        if p == 'first':
            tmp = tmp.sort_values(['serie', 'timestamp', 'lote']).groupby(['serie', 'timestamp'], as_index=False).first()
            tmp['resid'] = tmp['y_true'] - tmp['y_pred']
        elif p == 'last':
            tmp = tmp.sort_values(['serie', 'timestamp', 'lote']).groupby(['serie', 'timestamp'], as_index=False).last()
            tmp['resid'] = tmp['y_true'] - tmp['y_pred']
        else:
            med = tmp.groupby(['serie', 'timestamp'], as_index=False)['y_pred'].median().rename(columns={'y_pred': 'y_pred_eval'})
            tmp = tmp.merge(med, on=['serie', 'timestamp'], how='left')
            tmp['resid'] = tmp['y_true'] - tmp['y_pred_eval']

        tmp['ae'] = tmp['resid'].abs()
        tmp['se'] = tmp['resid'] ** 2
        m = tmp.groupby(['serie', 'lote'], as_index=False).agg(MAE=('ae', 'mean'), MSE=('se', 'mean')).sort_values(['serie', 'lote'])
        agg = m.groupby('lote', as_index=False).agg(MAE=('MAE', 'mean'), MSE=('MSE', 'mean'))
        agg['serie'] = 'AGREGADO'
        out[p] = pd.concat([m, agg], ignore_index=True)
    return out

def plot_metric_lines(metric_df: pd.DataFrame, metric: str, title: str):
    plt.figure(figsize=(12, 5))
    for serie, dfi in metric_df.groupby('serie'):
        plt.plot(dfi['lote'], dfi[metric], label=serie, alpha=0.9)
    plt.title(title)
    plt.xlabel('Lote / Janela de previsão')
    plt.ylabel(metric)
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout(); plt.show()

def plot_boxplots(metric_df: pd.DataFrame, metric: str, title: str):
    series_order = sorted(metric_df['serie'].unique())
    data = [metric_df.loc[metric_df['serie'] == s, metric].dropna().values for s in series_order]
    plt.figure(figsize=(max(8, len(series_order) * 0.7), 5))
    plt.boxplot(data, labels=series_order, showfliers=True)
    plt.title(title); plt.ylabel(metric)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout(); plt.show()

def plot_series_real_vs_pred(views: pd.DataFrame, pred_kind: str = 'median'):
    col = f'y_pred_{pred_kind}'
    for serie, dfi in views.groupby('serie'):
        dfi = dfi.sort_values('timestamp')
        plt.figure(figsize=(12, 4))
        plt.plot(dfi['timestamp'], dfi['y_true'], color='black', label='Série observada')
        plt.plot(dfi['timestamp'], dfi[col], color='red', label=f'Predição ({pred_kind})')
        plt.title(f'Série {serie}: observado vs predição ({pred_kind})')
        plt.xlabel('Tempo'); plt.ylabel('Valor')
        plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
raw = load_forecasts_csvs(PREVISOES_DIR, CSV_PATTERN)
print('Linhas carregadas:', len(raw))
print('Séries:', raw['serie'].nunique())
raw.head()

In [ ]:
views = add_prediction_views(raw)
views.head()

In [ ]:
metrics = rolling_metrics_by_batch(raw)
for paradigm, dfm in metrics.items():
    print(f'===== {paradigm.upper()} =====')
    display(dfm.head(20))

In [ ]:
for paradigm, dfm in metrics.items():
    plot_metric_lines(dfm, 'MAE', f'MAE por lote - paradigma {paradigm}')
    plot_metric_lines(dfm, 'MSE', f'MSE por lote - paradigma {paradigm}')

In [ ]:
for paradigm, dfm in metrics.items():
    plot_boxplots(dfm, 'MAE', f'Box-plot MAE - paradigma {paradigm}')
    plot_boxplots(dfm, 'MSE', f'Box-plot MSE - paradigma {paradigm}')

In [ ]:
plot_series_real_vs_pred(views, pred_kind='median')  # troque para first/median/last